# 과제 2 — 학습된 모델로 감성 예측
최고 성능 모델(Exp6 MPNet)을 사용하여 사용자 입력 문장의 감성을 예측

In [ ]:
# Cell 0: 패키지 설치
# sentence-transformers: MPNet 임베딩 모델 사용
!pip install sentence-transformers torch -q

In [ ]:
# Cell 1: 패키지 임포트
from sentence_transformers import SentenceTransformer
import torch
import torch.nn as nn

# GPU 사용 가능 시 GPU, 아니면 CPU 사용
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 디바이스: {device}')

In [ ]:
# Cell 2: MLP 클래스 정의
# 학습 시 사용한 동일한 구조의 신경망 모델
class MLP(nn.Module):
    """Multi-Layer Perceptron 분류 모델
    
    Args:
        input_size: 입력 벡터 차원 (MPNet의 경우 768)
        hidden_size: 첫 번째 은닉층 크기 (최적값: 1000)
        output_size: 출력 클래스 수 (3: Negative/Neutral/Positive)
        dropout_rate: 과적합 방지를 위한 Dropout 비율 (최적값: 0.2)
    """
    def __init__(self, input_size, hidden_size, output_size, dropout_rate=0.0):
        super().__init__()
        # 첫 번째 은닉층: input_size → hidden_size
        self.fc1 = nn.Linear(input_size, hidden_size)
        # 두 번째 은닉층: hidden_size → hidden_size//2
        self.fc2 = nn.Linear(hidden_size, hidden_size // 2)
        # 출력층: hidden_size//2 → output_size
        self.fc3 = nn.Linear(hidden_size // 2, output_size)
        # 활성화 함수: GELU (Gaussian Error Linear Unit)
        self.activation = nn.GELU()
        # 출력 활성화 함수: Softmax (확률 분포로 변환)
        self.output_act = nn.Softmax(dim=1)
        # Dropout: 학습 시 일부 뉴런을 무작위로 비활성화하여 과적합 방지
        self.dropout = nn.Dropout(p=dropout_rate)
    
    def forward(self, x):
        """순전파 (Forward Pass)
        
        Args:
            x: 입력 텐서 (batch_size, input_size)
        
        Returns:
            출력 텐서 (batch_size, output_size) - 각 클래스에 대한 확률
        """
        # 첫 번째 은닉층 → 활성화 → Dropout
        x = self.dropout(self.activation(self.fc1(x)))
        # 두 번째 은닉층 → 활성화 → Dropout
        x = self.dropout(self.activation(self.fc2(x)))
        # 출력층 → Softmax (확률로 변환)
        return self.output_act(self.fc3(x))

print('✅ MLP 클래스 정의 완료')

In [ ]:
# Cell 3: MPNet 임베딩 모델 로드
# 사전학습된 sentence-transformers 모델을 사용하여 문장을 768차원 벡터로 변환
print('MPNet 임베딩 모델 로드 중... (~420MB)')
embedder = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
print('✅ MPNet 로드 완료')
print(f'   출력 차원: 768')

In [ ]:
# Cell 4: 학습된 MLP 체크포인트 로드
# ⚠️ Colab Files 패널에서 'best_model_exp6.pt' 업로드 필요

# 모델 초기화 (학습 시와 동일한 하이퍼파라미터)
INPUT_SIZE = 768      # MPNet 출력 차원
HIDDEN_SIZE = 1000    # W&B Sweep으로 찾은 최적값
OUTPUT_SIZE = 3       # 클래스 수 (Negative/Neutral/Positive)
DROPOUT = 0.2         # W&B Sweep으로 찾은 최적값

model = MLP(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE, DROPOUT).to(device)

# 학습된 가중치 로드
checkpoint_path = 'best_model_exp6.pt'
model.load_state_dict(torch.load(checkpoint_path, map_location=device))

# 평가 모드로 전환 (Dropout 비활성화, BatchNorm 고정)
model.eval()

print('✅ 학습된 모델 로드 완료')
print(f'   체크포인트: {checkpoint_path}')
print(f'   하이퍼파라미터: hidden={HIDDEN_SIZE}, dropout={DROPOUT}')

In [ ]:
# Cell 5: 레이블 매핑 정의
# 모델 출력 (0, 1, 2) → 실제 감성 레이블
# ⚠️ 데이터셋 확인 후 실제 매핑에 맞게 수정 필요
label_map = {
    0: "Negative",  # 부정
    1: "Neutral",   # 중립
    2: "Positive"   # 긍정
}

print('✅ 레이블 매핑:')
for idx, label in label_map.items():
    print(f'   {idx} → {label}')

In [ ]:
# Cell 6: 예측 함수 정의
def predict_sentiment(sentence):
    """입력 문장의 감성을 예측하는 함수
    
    처리 과정:
    1. 문장을 MPNet으로 임베딩 (text → 768차원 벡터)
    2. 벡터를 PyTorch Tensor로 변환
    3. MLP 모델에 입력하여 예측
    4. Softmax 출력에서 가장 높은 확률의 클래스 선택
    5. 결과 출력
    
    Args:
        sentence (str): 예측할 문장
    
    Returns:
        None (결과를 직접 출력)
    """
    # Step 1: 문장 임베딩
    # MPNet이 문장을 의미 벡터(768차원)로 변환
    embedding = embedder.encode([sentence], convert_to_numpy=True)
    
    # Step 2: NumPy 배열 → PyTorch Tensor 변환
    # GPU 사용 시 GPU 메모리로 이동
    tensor = torch.FloatTensor(embedding).to(device)
    
    # Step 3: 모델 예측 (순전파)
    # torch.no_grad(): 기울기 계산 비활성화 (예측 시 불필요, 메모리 절약)
    with torch.no_grad():
        # 모델 출력: (batch_size=1, output_size=3) 형태의 확률 분포
        output = model(tensor)
        # argmax: 가장 높은 확률을 가진 클래스의 인덱스 선택
        pred_class = torch.argmax(output, dim=1).item()
    
    # Step 4: 예측 결과 출력
    sentiment = label_map[pred_class]
    print(f"Input sentence: {sentence}")
    print(f"This sentence is {sentiment} sentence.")

print('✅ 예측 함수 정의 완료')

In [ ]:
# Cell 7: 예측 테스트
# 다양한 감성의 문장으로 모델 성능 확인

print('='*60)
print('📝 감성 예측 테스트')
print('='*60)
print()

# 긍정 문장
predict_sentiment("I love this item")
print()

# 부정 문장
predict_sentiment("This is the worst product ever")
print()

# 중립 문장
predict_sentiment("It was okay, nothing special")
print()

print('='*60)

In [ ]:
# Cell 8: 사용자 입력 (Interactive)
# 직접 문장을 입력하여 예측 결과 확인

print('💬 사용자 입력 모드')
print('   입력 예시: "The movie was absolutely amazing!"')
print('   종료: "quit" 입력')
print()

while True:
    user_input = input('문장 입력: ').strip()
    
    if user_input.lower() == 'quit':
        print('종료합니다.')
        break
    
    if not user_input:
        print('⚠️ 문장을 입력해주세요.\n')
        continue
    
    predict_sentiment(user_input)
    print()